# Logistic Regression with Regularization
This notebook demonstrates a full workflow for logistic regression, including regularization, threshold tuning, and evaluation. Each step is explained with practical notes and intuitions.

## Why Logistic Regression?
- Logistic regression is a fundamental algorithm for binary classification.
- It models the probability of class membership using the logistic (sigmoid) function.
- Regularization (Ridge, Lasso, ElasticNet) helps prevent overfitting and can perform feature selection.

In [1]:
# 1. Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve
)

## Step 1: Data Preparation
We use synthetic data for demonstration. Replace with your own data for real projects.

In [ ]:
# 2. Sample Data
df = pd.DataFrame({
    'x1': np.random.rand(200),
    'x2': np.random.rand(200),
    'x3': np.random.rand(200),
    'y': np.random.randint(0, 2, 200)
})
X = df[['x1', 'x2', 'x3']]
y = df['y']
df.head()

### Notes:
- Features should be numeric.
- Always inspect your data for class balance and outliers.

In [ ]:
# 3. Train-Test Split
# Stratify ensures class balance in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train size:', len(X_train), '| Test size:', len(X_test))

### Notes:
- Never train and test on the same data!
- Stratification is crucial for fair evaluation, especially with imbalanced data.

In [ ]:
# 4. Base Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print('=== BASE MODEL ===')
print(classification_report(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_proba))

### Notes:
- ROC-AUC measures the ability to distinguish between classes.
- Use classification report for precision, recall, and F1-score.

In [ ]:
# 5. Threshold Tuning
# Default threshold is 0.5. Adjust to balance precision and recall.
threshold = 0.3  # lower threshold increases recall
y_pred_custom = (y_proba >= threshold).astype(int)
print('=== CUSTOM THRESHOLD ===')
print(classification_report(y_test, y_pred_custom))

### Notes:
- Lowering the threshold increases recall but may reduce precision.
- Tune threshold based on business needs (e.g., minimize false negatives).

In [ ]:
# 6. Ridge (L2 Regularization)
ridge = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000)
params = {'C': [0.01, 0.1, 1, 10]}  # C = 1/lambda
ridge_cv = GridSearchCV(ridge, params, cv=5)
ridge_cv.fit(X_train, y_train)
print('=== RIDGE ===')
print('Best C:', ridge_cv.best_params_)

### Notes:
- Ridge (L2) shrinks coefficients but keeps all features.
- Use when you suspect multicollinearity.

In [ ]:
# 7. Lasso (L1 Regularization)
lasso = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000)
params = {'C': [0.01, 0.1, 1, 10]}
lasso_cv = GridSearchCV(lasso, params, cv=5)
lasso_cv.fit(X_train, y_train)
print('=== LASSO ===')
print('Best C:', lasso_cv.best_params_)
print('Coefficients:', lasso_cv.best_estimator_.coef_)

### Notes:
- Lasso (L1) can set some coefficients to zero (feature selection).
- Use for sparse models or when you want to identify important features.

In [ ]:
# 8. Elastic Net (L1 + L2)
elastic = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=1000)
params = {
    'C': [0.01, 0.1, 1],
    'l1_ratio': [0.2, 0.5, 0.8]
}
elastic_cv = GridSearchCV(elastic, params, cv=5)
elastic_cv.fit(X_train, y_train)
print('=== ELASTIC NET ===')
print('Best params:', elastic_cv.best_params_)

### Notes:
- Elastic Net combines L1 and L2 penalties.
- Useful when you want both feature selection and coefficient shrinkage.

In [ ]:
# 9. Precision-Recall Curve
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
import matplotlib.pyplot as plt
plt.plot(thresholds, precision[:-1], label='Precision')
plt.plot(thresholds, recall[:-1], label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision-Recall vs Threshold')
plt.legend()
plt.show()

### Notes:
- Use the precision-recall curve to choose the best threshold for your application.
- Balance between false positives and false negatives based on business needs.

## Summary
- Logistic regression uses log-loss (not MSE).
- Regularization: Ridge (L2) shrinks coefficients, Lasso (L1) selects features, ElasticNet mixes both.
- Threshold tuning is critical for imbalanced data.
- Always evaluate using precision, recall, ROC-AUC, and PR-AUC.